In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import subprocess; subprocess.run(['pip', 'install', '-q', '--upgrade', 'openpyxl'])

import json, time
import requests
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ── Config ────────────────────────────────────────────────────────────────────
API_KEY          = 'neom-9172-sjou-mzxw'
BASE_URL         = 'https://api.boxtobox.ai/v1/wy/api/v3'
COMPETITION_WYID = 364
SEASON_ID        = 191622
SLEEP_SECONDS    = 0.15
EVENTS_DIR       = Path('/content/drive/MyDrive/Event data/Wyscout/Premier League/2025-2026')
OUT_XLSX         = '/content/drive/MyDrive/Event data/Wyscout/epl.xlsx'
COMPETITIONS_CSV = '/content/drive/MyDrive/Event data/Wyscout/domestic_competitions_all_areas.csv'

# ── Style constants ───────────────────────────────────────────────────────────
HEADER_BG     = '1F3864'
ID_BG         = 'D6E4F0'
ALT_BG        = 'F2F7FB'
ERR_BG        = 'FFF2CC'
BORDER        = Border(**{s: Side(style='thin', color='BDD7EE') for s in ('left','right','top','bottom')})
IDENTITY_COLS = {'playerId','playerName','teamName','positions_codes','competitionId','competitionName'}

def _hdr(cell, v):
    cell.value     = v
    cell.font      = Font(name='Calibri', bold=True, color='FFFFFF', size=11)
    cell.fill      = PatternFill('solid', fgColor=HEADER_BG)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border    = BORDER

def _cell(cell, v, bg, align='right', bold=False, num_fmt=None):
    cell.value     = v
    cell.fill      = PatternFill('solid', fgColor=bg)
    cell.font      = Font(name='Calibri', size=10, bold=bold)
    cell.alignment = Alignment(horizontal=align, vertical='center')
    cell.border    = BORDER
    if num_fmt: cell.number_format = num_fmt

def _width(series, header, cap=30):
    mx = series.dropna().astype(str).str.len().max() if not series.dropna().empty else 0
    return min(max(len(str(header)), mx) + 2, cap)

def _numfmt(col, series):
    low = col.lower()
    if 'id' in low: return '0'
    if any(k in low for k in ('percent','pct','rate','ratio','avg','mean')): return '0.00'
    if pd.api.types.is_numeric_dtype(series):
        if series.dropna().apply(float.is_integer).all() if not series.dropna().empty else True: return '0'
        return '0.00'
    return 'General'

def fmt_stats(ws, df):
    cols = list(df.columns)
    for ci, c in enumerate(cols, 1): _hdr(ws.cell(1, ci), c)
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        alt = ri % 2 == 0
        for ci, c in enumerate(cols, 1):
            v = row[c]; v = v.item() if hasattr(v, 'item') else v
            if c in IDENTITY_COLS:
                _cell(ws.cell(ri, ci), v, ID_BG, align='left', bold=(c=='playerName'))
            else:
                _cell(ws.cell(ri, ci), v, ALT_BG if alt else 'FFFFFF',
                      num_fmt=_numfmt(c, df[c]) if pd.api.types.is_numeric_dtype(df[c]) else None)
    for ci, c in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = _width(df[c], c)
    id_n = sum(1 for c in cols if c in IDENTITY_COLS)
    ws.freeze_panes = f'{get_column_letter(id_n+1)}2'
    ws.row_dimensions[1].height = 36
    for r in range(2, len(df)+2): ws.row_dimensions[r].height = 16
    if len(df):
        t = Table(displayName='AdvancedStats', ref=f'A1:{get_column_letter(len(cols))}{len(df)+1}')
        t.tableStyleInfo = TableStyleInfo(name='TableStyleMedium9', showRowStripes=True)
        ws.add_table(t)
    ws.title = 'Advanced Stats'

def fmt_failures(ws, df):
    for ci, c in enumerate(df.columns, 1): _hdr(ws.cell(1, ci), c)
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, c in enumerate(df.columns, 1):
            v = row[c]; v = v.item() if hasattr(v, 'item') else v
            _cell(ws.cell(ri, ci), v, ERR_BG, align='left')
    for ci, c in enumerate(df.columns, 1):
        ws.column_dimensions[get_column_letter(ci)].width = _width(df[c], c)
    ws.freeze_panes = 'A2'
    ws.title = 'Failures'

# ── Build player lookup ───────────────────────────────────────────────────────
print('Building player lookup...')
rows = []
for fp in EVENTS_DIR.glob('*.json'):
    try:
        data = json.loads(fp.read_text('utf-8'))
        for ev in data.get('events', []):
            p = ev.get('player', {}); t = ev.get('team', {})
            if p.get('id'): rows.append({'playerId': int(p['id']), 'playerName': p.get('name'), 'teamName': t.get('name')})
    except Exception: pass

df = pd.DataFrame(rows)
lookup = (
    df.groupby(['playerId','playerName']).size().reset_index(name='n')
      .sort_values(['playerId','n'], ascending=[True,False]).drop_duplicates('playerId')[['playerId','playerName']]
      .merge(
          df.groupby(['playerId','teamName']).size().reset_index(name='n')
            .sort_values(['playerId','n'], ascending=[True,False]).drop_duplicates('playerId')[['playerId','teamName']],
          on='playerId', how='left'
      )
)
print(f'  {len(lookup)} unique players found')

# ── Fetch advanced stats ──────────────────────────────────────────────────────
stat_rows, failures = [], []
player_ids = sorted(lookup['playerId'].unique())

for i, pid in enumerate(player_ids, 1):
    print(f'[{i}/{len(player_ids)}] player {pid}', end='\r')
    try:
        r = requests.get(
            f'{BASE_URL}/players/{pid}/advancedstats',
            params={'api_key': API_KEY, 'compId': COMPETITION_WYID, 'seasonId': SEASON_ID},
            timeout=30
        )
        r.raise_for_status()
        data = r.json()
        if isinstance(data, dict) and 'error' in data:
            failures.append({'playerId': pid, 'error': data['error']}); continue
        flat = pd.json_normalize(data, sep='.')
        flat['playerId'] = pid
        pos = data.get('positions') if isinstance(data, dict) else None
        if isinstance(pos, list):
            codes = [p.get('position',{}).get('code','').upper() for p in
                     sorted(pos, key=lambda x: x.get('percent',0), reverse=True)
                     if p.get('position',{}).get('code')]
            flat['positions_codes'] = ','.join(codes) if codes else None
        stat_rows.append(flat)
    except Exception as e:
        failures.append({'playerId': pid, 'error': str(e)})
    time.sleep(SLEEP_SECONDS)

print(f'\nDone — {len(stat_rows)} fetched, {len(failures)} failures')

# ── Merge & add competition name ──────────────────────────────────────────────
df_stats = pd.concat(stat_rows, ignore_index=True) if stat_rows else pd.DataFrame()
df_final = lookup.merge(df_stats, on='playerId', how='left')

comps = pd.read_csv(COMPETITIONS_CSV)
cid   = next((c for c in ['competitionId','wyId','id'] if c in comps.columns), None)
comps = comps.rename(columns={cid: 'competitionId', 'name': 'competitionName'})
comps['competitionId'] = pd.to_numeric(comps['competitionId'], errors='coerce')
if 'competitionId' not in df_final.columns: df_final['competitionId'] = COMPETITION_WYID
df_final['competitionId'] = pd.to_numeric(df_final['competitionId'], errors='coerce')
extra = [c for c in ['area.name','areaName','country','countryName'] if c in comps.columns]
df_final = df_final.merge(comps[['competitionId','competitionName']+extra].drop_duplicates('competitionId'),
                          on='competitionId', how='left')

front = [c for c in ['playerId','playerName','teamName','positions_codes','competitionId','competitionName']
         if c in df_final.columns]
df_final = df_final[front + sorted(c for c in df_final.columns if c not in front)]

# ── Write & format Excel ──────────────────────────────────────────────────────
df_fail = pd.DataFrame(failures)

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as w:
    df_final.to_excel(w, sheet_name='advancedstats', index=False)
    if not df_fail.empty: df_fail.to_excel(w, sheet_name='failures', index=False)

wb = load_workbook(OUT_XLSX)
fmt_stats(wb['advancedstats'], df_final)
if not df_fail.empty and 'failures' in wb.sheetnames: fmt_failures(wb['failures'], df_fail)
wb.save(OUT_XLSX)

print(f'Saved → {OUT_XLSX}')
df_final.head()